# Griffin Nsight Systems — Cross-Scenario Profiling Dashboard

This notebook regenerates a cross-scenario GPU profiling dashboard for Griffin's
realistic-scale nsys runs (train, finetune, inference) from raw analysis artifacts.

**Data source**: `artifacts/profiles/analysis/{run_id}/metrics.json` for each of
the 9 realistic slices.  `profiling/RESULTS.md` is used *only* as a validation
target in Section 8 — it is not the primary data source.

**Sections**
1. Configuration
2. Artifact discovery
3. Kernel time-share dashboard
4. Launch-count dashboard
5. NVTX region dashboard
6. Host / CUDA API overhead
7. Advisor summary cards
8. Optimization opportunity matrix
9. Validation against `RESULTS.md`
10. Experiment hook: next profiling question
11. Optional export

In [ ]:
from pathlib import Path

# ── Repo root auto-detection ─────────────────────────────────────────────────
_cwd = Path.cwd().resolve()
REPO_ROOT = next(
    (p for p in [_cwd, *_cwd.parents]
     if (p / "profiling").exists() and (p / "artifacts").exists()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError(
        "Cannot locate the Griffin repo root from the current working directory. "
        "Open this notebook from within the repo."
    )

ANALYSIS_DIR = REPO_ROOT / "artifacts" / "profiles" / "analysis"
NSYS_DIR     = REPO_ROOT / "artifacts" / "profiles" / "nsys"
EXPORT_DIR   = REPO_ROOT / "artifacts" / "profiles" / "dashboard_exports"

# ── Expected realistic runs ──────────────────────────────────────────────────
EXPECTED_RUNS = {
    "train": [
        "20260312-1526-trb1-realistic-20260312a-s01",
        "20260312-1532-trb1-realistic-20260312a-s02",
        "20260312-1537-trb1-realistic-20260312a-s03",
    ],
    "finetune": [
        "20260306-1538-ftb1-realistic-20260306a-s01",
        "20260306-1543-ftb1-realistic-20260306a-s02",
        "20260306-1553-ftb1-realistic-20260306a-s03",
    ],
    "inference": [
        "20260306-1647-ifb1-realistic-20260306b-s01",
        "20260306-1650-ifb1-realistic-20260306b-s02",
        "20260306-1652-ifb1-realistic-20260306b-s03",
    ],
}

TOP_K            = 10      # kernels per table / chart
EXPORT_ARTIFACTS = False   # set True to write CSVs to EXPORT_DIR

# ── Reference claims from RESULTS.md  (used only in Section 9) ───────────────
RESULTS_MD_TOP3 = [
    "ampere_sgemm_32x32_sliced1x4_tn",
    "fmha_cutlassF_f32_aligned_64x64_rf_sm80",
    "ampere_sgemm_32x128_tn",
]
RESULTS_MD_SCENARIO_PCT = {
    "train":     [30.4, 12.5,  9.3],
    "finetune":  [30.5, 12.5,  9.4],
    "inference": [31.7, 13.0,  9.8],
}
RESULTS_MD_SGEMM_LAUNCHES = {
    "train":     22731,
    "finetune":  22754,
    "inference":  7477,
}

print(f"Repo root : {REPO_ROOT}")
print(f"Analysis  : {ANALYSIS_DIR}")
print(f"NSys dir  : {NSYS_DIR}")

In [ ]:
import json
import re
import warnings

import numpy as np
import pandas as pd
from IPython.display import display, Markdown, HTML

pd.set_option("display.max_colwidth", 100)
pd.set_option("display.float_format", lambda v: f"{v:.3f}")

try:
    import plotly.express as px
    import plotly.graph_objects as go
    HAS_PLOTLY = True
except ImportError:
    HAS_PLOTLY = False
    import matplotlib
    import matplotlib.pyplot as plt
    import matplotlib.colors as mcolors
    try:
        plt.style.use("seaborn-v0_8-whitegrid")
    except OSError:
        plt.style.use("seaborn-whitegrid")

_lib = "Plotly (interactive)" if HAS_PLOTLY else "matplotlib (static fallback)"
display(Markdown(f"**Visualization library**: {_lib}"))

SCENARIO_COLORS = {"train": "#4C72B0", "finetune": "#55A868", "inference": "#C44E52"}
SCENARIO_ORDER  = ["train", "finetune", "inference"]

In [ ]:
import re, json
import numpy as np
import pandas as pd

# ── Kernel name utilities ─────────────────────────────────────────────────────

def kernel_base_name(name: str) -> str:
    # Strip template/arg params and trailing unicode ellipsis for stable grouping.
    name = re.sub(r'[<(].*', '', name).strip().rstrip('\u2026').strip()
    return name

_DISPLAY_OVERRIDES = {
    "ampere_sgemm_32x32_sliced1x4_tn":                "sgemm_32x32_tn  [small GEMM]",
    "fmha_cutlassF_f32_aligned_64x64_rf_sm80":         "fmha_f32_64x64  [attention]",
    "ampere_sgemm_32x128_tn":                          "sgemm_32x128_tn  [GEMM]",
    "ampere_sgemm_64x32_sliced1x4_tn":                 "sgemm_64x32_tn",
    "ampere_sgemm_128x64_tn":                          "sgemm_128x64_tn",
    "ampere_sgemm_64x64_tn":                           "sgemm_64x64_tn",
    "void cublasLt::splitKreduce_kernel":              "splitKreduce  [reduction]",
    "void at::native::elementwise_kernel":             "elementwise_kernel",
    "void at::native::vectorized_elementwise_kernel":  "vectorized_eltwise",
    "void at::native::vectorized_layer_norm_kernel":   "layer_norm_kernel",
    "std::enable_if<!T7":                              "gemvx_kernel",
}

def kernel_label(base: str) -> str:
    return _DISPLAY_OVERRIDES.get(base, base[:55])

# ── Data loader ───────────────────────────────────────────────────────────────

def load_run_data(run_id: str) -> dict:
    # Try metrics.json first; fall back to reporting missing artifact clearly.
    json_path   = ANALYSIS_DIR / run_id / "metrics.json"
    sqlite_path = NSYS_DIR / f"{run_id}.sqlite"
    nsys_path   = NSYS_DIR / f"{run_id}.nsys-rep"

    status = {
        "run_id":          run_id,
        "json_exists":     json_path.exists(),
        "sqlite_exists":   sqlite_path.exists(),
        "nsys_rep_exists": nsys_path.exists(),
        "source":          None,
        "parse_status":    "pending",
        "notes":           "",
        "data":            None,
    }

    if json_path.exists():
        try:
            status["data"]         = json.loads(json_path.read_text())
            status["source"]       = "metrics.json"
            status["parse_status"] = "ok"
        except Exception as exc:
            status["source"]       = "metrics.json"
            status["parse_status"] = "parse_error"
            status["notes"]        = str(exc)
    elif sqlite_path.exists():
        status["source"]       = "sqlite"
        status["parse_status"] = "missing_metrics_json"
        status["notes"]        = (
            "SQLite present but no metrics.json. "
            "Regenerate with: bash scripts/analyze_nsys_run.sh " + run_id
        )
    elif nsys_path.exists():
        status["source"]       = "nsys-rep"
        status["parse_status"] = "missing_analysis_bundle"
        status["notes"]        = (
            "nsys-rep present but no analysis bundle. "
            "Run: bash scripts/analyze_nsys_run.sh " + run_id
        )
    else:
        status["parse_status"] = "missing_all"
        status["notes"]        = "No artifact found for this run ID."

    return status

# ── DataFrame builders ────────────────────────────────────────────────────────

def build_kernel_df(run_data_dict: dict) -> pd.DataFrame:
    rows = []
    for run_id, info in run_data_dict.items():
        for rank, k in enumerate(info["data"].get("top_gpu_kernels", []), 1):
            base = kernel_base_name(k["kernel_name"])
            rows.append({
                "scenario":     info["scenario"],
                "slice":        info["slice_idx"],
                "run_id":       run_id,
                "rank":         rank,
                "kernel_name":  k["kernel_name"],
                "kernel_base":  base,
                "kernel_label": kernel_label(base),
                "launches":     k["launches"],
                "total_ms":     k["total_ms"],
                "avg_us":       k["avg_us"],
                "time_pct":     k["time_pct"],
            })
    return pd.DataFrame(rows)

def build_nvtx_df(run_data_dict: dict) -> pd.DataFrame:
    rows = []
    for run_id, info in run_data_dict.items():
        for region in info["data"].get("top_nvtx_ranges", []):
            label = region["label"].replace("PushPop  ", "").replace("PushPop ", "").strip()
            rows.append({
                "scenario":    info["scenario"],
                "slice":       info["slice_idx"],
                "run_id":      run_id,
                "nvtx_label":  label,
                "instances":   region["instances"],
                "total_ms":    region["total_ms"],
                "avg_us":      region["avg_us"],
                "time_pct":    region["time_pct"],
                "nvtx_status": info["data"].get("nvtx_coverage_status", "unknown"),
            })
    return pd.DataFrame(rows)

def build_api_df(run_data_dict: dict) -> pd.DataFrame:
    rows = []
    for run_id, info in run_data_dict.items():
        for api in info["data"].get("top_cuda_runtime_calls", []):
            rows.append({
                "scenario": info["scenario"],
                "slice":    info["slice_idx"],
                "run_id":   run_id,
                "api_name": api["api_name"],
                "calls":    api["calls"],
                "total_ms": api["total_ms"],
                "avg_us":   api["avg_us"],
                "time_pct": api["time_pct"],
            })
    return pd.DataFrame(rows)

print("Helper functions defined.")

In [ ]:
all_run_data = {}
for scenario, run_ids in EXPECTED_RUNS.items():
    for idx, run_id in enumerate(run_ids, 1):
        result              = load_run_data(run_id)
        result["scenario"]  = scenario
        result["slice_idx"] = idx
        all_run_data[run_id] = result

loaded_runs = {k: v for k, v in all_run_data.items() if v["parse_status"] == "ok"}
failed_runs = {k: v for k, v in all_run_data.items() if v["parse_status"] != "ok"}

short_label = {
    run_id: f"{info['scenario'][0].upper()}{info['slice_idx']}"
    for run_id, info in all_run_data.items()
}

print(f"Loaded: {len(loaded_runs)}/9 runs  |  Failed: {len(failed_runs)}/9 runs")
if failed_runs:
    for run_id, info in failed_runs.items():
        print(f"  [MISSING] {run_id}  ({info['parse_status']})")
        if info["notes"]:
            print(f"           {info['notes']}")

## 1. Artifact Discovery

The table below shows what files were found for each of the 9 expected realistic
runs.  Parse status `ok` means `metrics.json` was found and loaded successfully.

To regenerate a missing analysis bundle:
```bash
bash scripts/analyze_nsys_run.sh <run_id>
```

In [ ]:
rows = []
for scenario in SCENARIO_ORDER:
    for idx, run_id in enumerate(EXPECTED_RUNS[scenario], 1):
        info = all_run_data[run_id]
        rows.append({
            "scenario":     scenario,
            "slice":        idx,
            "run_id":       run_id,
            "nsys-rep":     "v" if info["nsys_rep_exists"] else "x",
            "sqlite":       "v" if info["sqlite_exists"]   else "x",
            "metrics.json": "v" if info["json_exists"]     else "x",
            "parse_status": info["parse_status"],
            "notes":        info["notes"] or "ok",
        })

avail_df = pd.DataFrame(rows)

def _color_status(v):
    if v == "ok":              return "color: green; font-weight: bold"
    if v in ("missing_all", "parse_error"): return "color: red"
    return "color: orange"

_applymap = getattr(avail_df.style, "map", None) or avail_df.style.applymap
display(_applymap(_color_status, subset=["parse_status"]))

## 2. Kernel Time-Share Dashboard

GPU kernel time-share data comes from `metrics.json → top_gpu_kernels` in each
analysis bundle.  Each run reports up to 15 kernels ranked by total GPU time %.

**Key question**: Do the same three kernels dominate all 9 slices across all 3
scenarios with identical rank ordering?

In [ ]:
if not loaded_runs:
    display(Markdown(
        "**No data loaded.** Run `scripts/analyze_nsys_run.sh` for each expected run ID."
    ))
    kernel_df = pd.DataFrame()
else:
    kernel_df = build_kernel_df(loaded_runs)
    print(
        f"kernel_df: {len(kernel_df)} rows | "
        f"{kernel_df['run_id'].nunique()} runs | "
        f"{kernel_df['kernel_base'].nunique()} unique kernel bases"
    )

In [ ]:
if not kernel_df.empty:
    display(Markdown(f"### Per-Run Top-{TOP_K} Kernels"))
    per_run = (
        kernel_df[kernel_df["rank"] <= TOP_K]
        [["scenario", "slice", "run_id", "rank", "kernel_base", "time_pct", "launches", "avg_us"]]
        .sort_values(["scenario", "slice", "rank"])
        .reset_index(drop=True)
    )
    for scenario in SCENARIO_ORDER:
        sub = per_run[per_run["scenario"] == scenario].drop(columns=["scenario"])
        display(Markdown(f"#### {scenario.capitalize()}"))
        display(sub.reset_index(drop=True))

In [ ]:
if not kernel_df.empty:
    scenario_agg = (
        kernel_df[kernel_df["rank"] <= TOP_K]
        .groupby(["scenario", "kernel_base", "kernel_label"], as_index=False)
        .agg(
            mean_time_pct  = ("time_pct",  "mean"),
            std_time_pct   = ("time_pct",  "std"),
            mean_launches  = ("launches",  "mean"),
            total_launches = ("launches",  "sum"),
            min_rank       = ("rank",      "min"),
            max_rank       = ("rank",      "max"),
            slices         = ("run_id",    "nunique"),
        )
        .sort_values(["scenario", "mean_time_pct"], ascending=[True, False])
    )

    all_top_kernels = (
        scenario_agg
        .groupby("kernel_base")["mean_time_pct"]
        .mean()
        .sort_values(ascending=False)
        .head(TOP_K)
        .index.tolist()
    )

    display(Markdown("### Per-Scenario Aggregated Top Kernels (mean across 3 slices)"))
    for scenario in SCENARIO_ORDER:
        sub = scenario_agg[scenario_agg["scenario"] == scenario].head(TOP_K)
        display(Markdown(f"#### {scenario.capitalize()}"))
        display(sub.drop(columns=["scenario"]).reset_index(drop=True).round(2))
else:
    scenario_agg    = pd.DataFrame()
    all_top_kernels = []

In [ ]:
if not scenario_agg.empty and all_top_kernels:
    plot_df = scenario_agg[scenario_agg["kernel_base"].isin(all_top_kernels)].copy()
    plot_df["kernel_label"] = plot_df["kernel_base"].apply(kernel_label)

    display(Markdown(f"### Top-{TOP_K} Kernels — Mean GPU Time % by Scenario"))
    if HAS_PLOTLY:
        fig = px.bar(
            plot_df.sort_values("mean_time_pct"),
            x="mean_time_pct",
            y="kernel_label",
            color="scenario",
            barmode="group",
            orientation="h",
            title=f"Top-{TOP_K} Kernels — Mean GPU Time % by Scenario",
            labels={
                "mean_time_pct": "Mean GPU Time %",
                "kernel_label":  "Kernel",
                "scenario":      "Scenario",
            },
            color_discrete_map=SCENARIO_COLORS,
            error_x="std_time_pct",
        )
        fig.update_layout(height=520, yaxis={"categoryorder": "total ascending"})
        fig.show()
    else:
        pivot = (
            plot_df.pivot_table(index="kernel_label", columns="scenario", values="mean_time_pct")
            .fillna(0)
            .reindex(columns=SCENARIO_ORDER, fill_value=0)
        )
        pivot = pivot.sort_values("train")
        fig, ax = plt.subplots(figsize=(13, 7))
        pivot.plot.barh(ax=ax, color=[SCENARIO_COLORS[s] for s in pivot.columns])
        ax.set_title(f"Top-{TOP_K} Kernels — Mean GPU Time % by Scenario")
        ax.set_xlabel("Mean GPU Time %")
        ax.legend(title="Scenario")
        plt.tight_layout()
        plt.show()
else:
    display(Markdown("*No kernel data available for chart.*"))

In [ ]:
if not kernel_df.empty and all_top_kernels:
    rank_matrix = (
        kernel_df[kernel_df["kernel_base"].isin(all_top_kernels)]
        .pivot_table(index="kernel_base", columns="run_id", values="rank", aggfunc="min")
    )
    ordered_runs = [r for s in SCENARIO_ORDER for r in EXPECTED_RUNS[s] if r in rank_matrix.columns]
    rank_matrix  = rank_matrix[ordered_runs]
    rank_matrix.columns = [short_label[c] for c in rank_matrix.columns]
    rank_matrix.index   = [kernel_label(k) for k in rank_matrix.index]

    display(Markdown(
        "### Rank Stability Heatmap\n\n"
        "Cell = rank in that run (1 = most dominant; blank = not in top-10).  "
        "Column groups: **T**=train, **F**=finetune, **I**=inference."
    ))

    if HAS_PLOTLY:
        z     = rank_matrix.values.astype(float)
        ztext = np.where(np.isnan(z), "", z.astype("Int64").astype(str))
        fig = go.Figure(go.Heatmap(
            z=z,
            x=rank_matrix.columns.tolist(),
            y=rank_matrix.index.tolist(),
            text=ztext,
            texttemplate="%{text}",
            colorscale="RdYlGn_r",
            showscale=True,
            colorbar={"title": "Rank"},
            zmin=1, zmax=TOP_K,
        ))
        fig.update_layout(
            title="Kernel Rank Stability — All 9 Slices",
            xaxis_title="Run (T=train · F=finetune · I=inference)",
            yaxis_title="Kernel",
            height=450,
        )
        fig.show()
    else:
        cmap = plt.cm.RdYlGn_r
        cmap.set_bad("lightgrey")
        masked = np.ma.masked_invalid(rank_matrix.values.astype(float))
        fig, ax = plt.subplots(figsize=(11, 5))
        im = ax.imshow(masked, cmap=cmap, aspect="auto", vmin=1, vmax=TOP_K)
        ax.set_xticks(range(len(rank_matrix.columns)))
        ax.set_xticklabels(rank_matrix.columns)
        ax.set_yticks(range(len(rank_matrix.index)))
        ax.set_yticklabels(rank_matrix.index, fontsize=8)
        plt.colorbar(im, ax=ax, label="Rank (lower=more dominant)")
        for i in range(rank_matrix.shape[0]):
            for j in range(rank_matrix.shape[1]):
                v = rank_matrix.values[i, j]
                if not np.isnan(v):
                    ax.text(j, i, str(int(v)), ha="center", va="center", fontsize=7)
        ax.set_title("Kernel Rank Stability Heatmap")
        plt.tight_layout()
        plt.show()

    display(rank_matrix.fillna("—"))
else:
    display(Markdown("*No kernel data available for rank stability chart.*"))

In [ ]:
if not kernel_df.empty:
    top3_rows = []
    for scenario in SCENARIO_ORDER:
        sub = kernel_df[kernel_df["scenario"] == scenario]
        for rank_val in [1, 2, 3]:
            r_sub = sub[sub["rank"] == rank_val]
            if r_sub.empty:
                top3_rows.append({
                    "scenario": scenario, "rank": rank_val,
                    "kernel_base": "NO DATA", "mean_time_pct": float("nan"),
                    "intra_scenario_stable": "unknown",
                })
                continue
            names     = r_sub["kernel_base"].unique().tolist()
            mean_pct  = r_sub["time_pct"].mean()
            is_stable = len(names) == 1
            top3_rows.append({
                "scenario":              scenario,
                "rank":                  rank_val,
                "kernel_base":           names[0] if is_stable else f"MIXED({len(names)})",
                "mean_time_pct":         round(mean_pct, 1),
                "intra_scenario_stable": "yes" if is_stable else "no",
            })

    top3_table = pd.DataFrame(top3_rows)
    display(Markdown("### Top-3 Kernel Identity and Rank — Cross-Scenario"))
    display(top3_table)

    display(Markdown("**Cross-scenario rank stability:**"))
    for rank_val in [1, 2, 3]:
        r_sub  = top3_table[top3_table["rank"] == rank_val]
        names  = r_sub["kernel_base"].unique().tolist()
        stable = "STABLE" if len(names) == 1 else f"MIXED ({names})"
        kernel = names[0] if len(names) == 1 else "---"
        display(Markdown(f"- Rank {rank_val}: **{stable}** — `{kernel}`"))
else:
    top3_table = pd.DataFrame()
    display(Markdown("*No kernel data available for top-3 table.*"))

## 3. Launch-Count Dashboard

A high launch count for a small-tile GEMM kernel suggests fragmentation — many
individually-launched small GEMMs.  The expected signature is that train/finetune
have ~3× the `sgemm_32x32` launch count of inference, because inference runs
eval-only with ~1/3 the graph evaluations.

In [ ]:
SGEMM32 = "ampere_sgemm_32x32_sliced1x4_tn"

if not kernel_df.empty:
    sgemm_launches = (
        kernel_df[kernel_df["kernel_base"] == SGEMM32]
        [["scenario", "slice", "run_id", "launches", "time_pct", "avg_us"]]
        .sort_values(["scenario", "slice"])
        .reset_index(drop=True)
    )

    display(Markdown(f"### `{SGEMM32}` — Per-Run Launch Count"))
    display(sgemm_launches)

    sgemm_scen_agg = (
        sgemm_launches.groupby("scenario")
        .agg(
            mean_launches = ("launches", "mean"),
            std_launches  = ("launches", "std"),
            mean_time_pct = ("time_pct", "mean"),
            mean_avg_us   = ("avg_us",   "mean"),
        )
        .reset_index()
        .round(1)
    )
    display(Markdown("### Per-Scenario sgemm_32x32 Summary"))
    display(sgemm_scen_agg)

    inf_row  = sgemm_scen_agg[sgemm_scen_agg["scenario"] == "inference"]
    if not inf_row.empty:
        inf_mean  = float(inf_row["mean_launches"].iloc[0])
        sgemm_mean_all = float(sgemm_launches["launches"].mean())
        ratios = []
        for _, row in sgemm_scen_agg.iterrows():
            if row["scenario"] != "inference" and inf_mean > 0:
                ratio = row["mean_launches"] / inf_mean
                ratios.append(
                    f"- **{row['scenario'].capitalize()} / inference**: {ratio:.2f}x  "
                    f"({row['mean_launches']:.0f} vs {inf_mean:.0f} launches/slice)"
                )
        if ratios:
            display(Markdown("### Scenario Launch-Count Ratio\n" + "\n".join(ratios)))
    else:
        sgemm_mean_all = float(sgemm_launches["launches"].mean()) if not sgemm_launches.empty else 0.0
else:
    sgemm_launches = pd.DataFrame()
    sgemm_scen_agg = pd.DataFrame()
    sgemm_mean_all = 0.0
    display(Markdown("*No kernel data available for launch count analysis.*"))

In [ ]:
if not kernel_df.empty and all_top_kernels:
    launch_agg = (
        kernel_df[kernel_df["kernel_base"].isin(all_top_kernels[:8])]
        .groupby(["scenario", "kernel_base"])["launches"]
        .mean()
        .reset_index()
        .rename(columns={"launches": "mean_launches_per_slice"})
    )
    launch_agg["kernel_label"] = launch_agg["kernel_base"].apply(kernel_label)

    display(Markdown("### Mean Launch Count per Slice — Top Kernels by Scenario (log scale)"))
    if HAS_PLOTLY:
        fig = px.bar(
            launch_agg,
            x="kernel_label",
            y="mean_launches_per_slice",
            color="scenario",
            barmode="group",
            log_y=True,
            title="Mean Launch Count per Slice (log scale)",
            labels={
                "mean_launches_per_slice": "Mean Launches / Slice",
                "kernel_label": "Kernel",
            },
            color_discrete_map=SCENARIO_COLORS,
        )
        fig.update_layout(height=450, xaxis_tickangle=-40)
        fig.show()
    else:
        pivot = (
            launch_agg.pivot_table(
                index="kernel_label", columns="scenario",
                values="mean_launches_per_slice"
            )
            .fillna(0)
            .reindex(columns=SCENARIO_ORDER, fill_value=0)
        )
        fig, ax = plt.subplots(figsize=(12, 5))
        pivot.plot.bar(ax=ax, color=[SCENARIO_COLORS[s] for s in pivot.columns], logy=True)
        ax.set_title("Mean Launch Count per Slice (log scale)")
        ax.set_ylabel("Mean Launches / Slice")
        ax.set_xlabel("")
        plt.xticks(rotation=40, ha="right")
        plt.tight_layout()
        plt.show()
else:
    display(Markdown("*No kernel data for launch count chart.*"))

## 4. NVTX Region Dashboard

NVTX regions annotate Griffin's execution phases (`gfm.eval_task`,
`gfm.train_epoch`, etc.).  Stable NVTX time-share across slices validates that
kernel-level time-share is a reliable signal.

> **If NVTX data is missing**: the section degrades gracefully and explains
> which export is needed.

In [ ]:
if not loaded_runs:
    nvtx_df  = pd.DataFrame()
    gfm_nvtx = pd.DataFrame()
    display(Markdown("**No data loaded — NVTX section unavailable.**"))
else:
    nvtx_df = build_nvtx_df(loaded_runs)

    if nvtx_df.empty:
        gfm_nvtx = pd.DataFrame()
        display(Markdown(
            "**Warning**: No NVTX data in any loaded metrics.json.  "
            "Ensure runs were captured with NVTX annotation and that "
            "`analyze_nsys_run.sh` was run with `nvtx_sum` enabled."
        ))
    else:
        missing_nvtx = nvtx_df[nvtx_df["nvtx_status"] != "present"]["run_id"].unique().tolist()
        if missing_nvtx:
            display(Markdown(
                f"**Warning**: {len(missing_nvtx)} run(s) have "
                f"`nvtx_coverage_status != present`: " + ", ".join(missing_nvtx)
            ))

        gfm_nvtx = nvtx_df[nvtx_df["nvtx_label"].str.startswith("gfm.")].copy()

        display(Markdown("### gfm.* NVTX Region Time Share — Mean % per Slice"))
        for scenario in SCENARIO_ORDER:
            sub = gfm_nvtx[gfm_nvtx["scenario"] == scenario]
            if sub.empty:
                display(Markdown(f"- **{scenario}**: No gfm.* NVTX data"))
                continue
            pivot = sub.pivot_table(
                index="nvtx_label", columns="slice", values="time_pct", aggfunc="mean"
            )
            pivot.columns  = [f"slice_{c}" for c in pivot.columns]
            pivot["mean"]  = pivot.mean(axis=1)
            display(Markdown(f"#### {scenario.capitalize()}"))
            display(pivot.round(1).sort_values("mean", ascending=False))

        display(Markdown("### Intra-Scenario NVTX Stability (std across slices)"))
        stab = (
            gfm_nvtx.groupby(["scenario", "nvtx_label"])["time_pct"]
            .agg(mean_pct="mean", std_pct="std")
            .reset_index()
            .round(2)
            .sort_values(["scenario", "mean_pct"], ascending=[True, False])
        )
        display(stab)

In [ ]:
_gfm_nvtx = gfm_nvtx if "gfm_nvtx" in dir() and not gfm_nvtx.empty else pd.DataFrame()

if not _gfm_nvtx.empty:
    nvtx_agg = (
        _gfm_nvtx.groupby(["scenario", "nvtx_label"])["time_pct"]
        .mean()
        .reset_index()
    )

    display(Markdown("### NVTX Time Share — Stacked by Scenario"))
    if HAS_PLOTLY:
        fig = px.bar(
            nvtx_agg,
            x="scenario",
            y="time_pct",
            color="nvtx_label",
            barmode="stack",
            title="NVTX Region Time Share by Scenario (gfm.* regions)",
            labels={"time_pct": "Mean NVTX Time %", "nvtx_label": "NVTX Region"},
            category_orders={"scenario": SCENARIO_ORDER},
        )
        fig.update_layout(height=450)
        fig.show()
    else:
        pivot = (
            nvtx_agg.pivot_table(index="scenario", columns="nvtx_label", values="time_pct")
            .fillna(0)
            .reindex(SCENARIO_ORDER)
        )
        fig, ax = plt.subplots(figsize=(9, 5))
        pivot.plot.bar(ax=ax, stacked=True)
        ax.set_title("NVTX Region Time Share by Scenario")
        ax.set_ylabel("Mean NVTX Time %")
        ax.legend(loc="upper right", fontsize=8, title="NVTX Region")
        plt.xticks(rotation=0)
        plt.tight_layout()
        plt.show()
else:
    display(Markdown(
        "*NVTX stacked bar unavailable — no `gfm.*` region data found.  "
        "Re-run slices with NVTX annotations to enable this section.*"
    ))

## 5. Host / CUDA API Overhead Dashboard

`cudaLaunchKernel` dominates CUDA API time (expected).  The interesting signal
is `cudaHostAlloc`: elevated in train/finetune (optimizer-state pinned-memory
allocation) but near-absent in inference (no optimizer), making it a secondary
overhead candidate worth quantifying.

In [ ]:
if not loaded_runs:
    api_df = pd.DataFrame()
    host_alloc_avg_us   = float("nan")
    host_alloc_elevated = False
    display(Markdown("**No data loaded — API overhead section unavailable.**"))
else:
    api_df = build_api_df(loaded_runs)

if not api_df.empty:
    display(Markdown("### Top CUDA Runtime API Calls by Scenario (mean across slices)"))
    api_agg = (
        api_df.groupby(["scenario", "api_name"], as_index=False)
        .agg(
            mean_calls    = ("calls",    "mean"),
            mean_time_pct = ("time_pct", "mean"),
            mean_avg_us   = ("avg_us",   "mean"),
        )
        .sort_values(["scenario", "mean_time_pct"], ascending=[True, False])
        .round(2)
    )
    for scenario in SCENARIO_ORDER:
        sub = api_agg[api_agg["scenario"] == scenario].drop(columns="scenario").head(8)
        display(Markdown(f"#### {scenario.capitalize()}"))
        display(sub.reset_index(drop=True))

    host_alloc_df = api_df[api_df["api_name"] == "cudaHostAlloc"]
    if not host_alloc_df.empty:
        display(Markdown("### `cudaHostAlloc` Overhead by Scenario"))
        ha_agg = (
            host_alloc_df.groupby("scenario")
            .agg(
                mean_calls    = ("calls",    "mean"),
                mean_avg_us   = ("avg_us",   "mean"),
                mean_time_pct = ("time_pct", "mean"),
            )
            .reset_index()
            .round(1)
        )
        display(ha_agg)
        host_alloc_avg_us   = float(host_alloc_df["avg_us"].mean())
        host_alloc_elevated = (
            host_alloc_df[host_alloc_df["scenario"].isin(["train", "finetune"])]["calls"].mean() > 100
        )
    else:
        display(Markdown("*`cudaHostAlloc` not found in top API calls for any run.*"))
        host_alloc_avg_us   = float("nan")
        host_alloc_elevated = False

    top_apis  = api_agg.groupby("api_name")["mean_time_pct"].max().nlargest(6).index.tolist()
    api_chart = api_agg[api_agg["api_name"].isin(top_apis)]
    display(Markdown("### Top CUDA API Calls — Time % by Scenario"))
    if HAS_PLOTLY:
        fig = px.bar(
            api_chart,
            x="api_name",
            y="mean_time_pct",
            color="scenario",
            barmode="group",
            title="Top CUDA API Calls — Mean Time % by Scenario",
            labels={"mean_time_pct": "Mean API Time %", "api_name": "API"},
            color_discrete_map=SCENARIO_COLORS,
        )
        fig.update_layout(height=420, xaxis_tickangle=-30)
        fig.show()
    else:
        pivot = (
            api_chart.pivot_table(
                index="api_name", columns="scenario", values="mean_time_pct"
            )
            .fillna(0)
            .reindex(columns=SCENARIO_ORDER, fill_value=0)
        )
        fig, ax = plt.subplots(figsize=(11, 5))
        pivot.plot.bar(ax=ax, color=[SCENARIO_COLORS[s] for s in pivot.columns])
        ax.set_title("Top CUDA API Calls — Mean Time %")
        ax.set_ylabel("Mean API Time %")
        plt.xticks(rotation=30, ha="right")
        plt.tight_layout()
        plt.show()
else:
    host_alloc_avg_us   = float("nan")
    host_alloc_elevated = False
    display(Markdown("*No CUDA API data available.*"))

## 6. Advisor Summary

The bullets below are **generated from computed tables**, not copied from
`RESULTS.md`.  They summarise the profiling evidence for a faculty advisor meeting.

In [ ]:
if kernel_df.empty:
    display(Markdown("**No data — advisor summary unavailable.**"))
    summary_lines = []
else:
    top1_series = (
        kernel_df[kernel_df["rank"] == 1]
        .groupby("kernel_base")["time_pct"]
        .mean()
        .sort_values(ascending=False)
    )
    dominant_kernel = top1_series.index[0] if not top1_series.empty else "unknown"
    dominant_pct    = float(top1_series.iloc[0]) if not top1_series.empty else 0.0

    rank_stable = True
    if not top3_table.empty:
        for _rv in [1, 2, 3]:
            _names = top3_table[top3_table["rank"] == _rv]["kernel_base"].unique()
            if len(_names) != 1:
                rank_stable = False
                break

    fmha_base     = "fmha_cutlassF_f32_aligned_64x64_rf_sm80"
    fmha_rows     = kernel_df[kernel_df["kernel_base"] == fmha_base]
    fmha_present  = not fmha_rows.empty
    fmha_pct      = float(fmha_rows["time_pct"].mean()) if fmha_present else 0.0

    _sm_all = sgemm_mean_all if "sgemm_mean_all" in dir() else 0.0
    _h_elev = host_alloc_elevated if "host_alloc_elevated" in dir() else False

    summary_lines = [
        f"- **Dominant GPU hotspot**: `{dominant_kernel}` averages "
        f"**{dominant_pct:.1f}%** GPU time across all 9 slices / all 3 scenarios.",

        f"- **Top-3 ranking**: "
        + ("**Stable** — identical kernel identity and rank order across all 3 scenarios."
           if rank_stable
           else "**NOT stable** — see top-3 cross-scenario table above."),

        (f"- **Flash attention**: Present — `{fmha_base}` averages **{fmha_pct:.1f}% GPU time** (rank 2 across all scenarios)."
         if fmha_present
         else "- **Flash attention**: Not detected in top-10 kernels."),

        f"- **Launch fragmentation**: `ampere_sgemm_32x32_sliced1x4_tn` averages "
        f"**{_sm_all:,.0f} launches/slice** — consistent with many repeated small-tile GEMM invocations.",

        ("- **Host allocation overhead**: Elevated `cudaHostAlloc` in train/finetune "
         "(optimizer-state allocation); near-absent in inference."
         if _h_elev
         else "- **Host allocation overhead**: Within expected range or not elevated."),

        "- **NCU deep dive**: Justified — TR-N1 bundle already captured for "
        f"`{dominant_kernel}`; TR-N2 approved for `fmha_cutlassF_f32_aligned_64x64_rf_sm80`.",

        "- **Next bounded experiment**: Confirm grid-fill efficiency for "
        f"`{dominant_kernel}` using existing NCU TR-N1 Waves/SM data; "
        "then collect matched NCU data for finetune/inference scenarios.",
    ]

    _html = (
        "<div style='background:#f7f7f7;border-left:4px solid #4C72B0;"
        "padding:14px 18px;border-radius:4px;font-size:14px;line-height:1.9'>"
        + "".join(f"<p style='margin:4px 0'>{ln}</p>" for ln in summary_lines)
        + "</div>"
    )
    display(HTML(_html))

## 7. Optimization Opportunity Matrix

Evidence levels: **high** = directly observed across multiple runs/scenarios;
**medium** = inferred from indirect signal; **low** = not yet observed or absent.

In [ ]:
_dom_pct   = dominant_pct        if "dominant_pct"       in dir() else float("nan")
_sgemm_aus = float(
    kernel_df[kernel_df["kernel_base"] == "ampere_sgemm_32x32_sliced1x4_tn"]["avg_us"].mean()
) if not kernel_df.empty else float("nan")
_fmha_pct  = fmha_pct            if "fmha_pct"           in dir() else float("nan")
_ha_us     = host_alloc_avg_us   if "host_alloc_avg_us"  in dir() else float("nan")
_sm_all    = sgemm_mean_all      if "sgemm_mean_all"     in dir() else float("nan")
_h_elev    = host_alloc_elevated if "host_alloc_elevated" in dir() else False
_splitK    = ("void cublasLt::splitKreduce_kernel" in kernel_df["kernel_base"].values
              if not kernel_df.empty else False)
_nvtx_ok   = ("gfm_nvtx" in dir() and not gfm_nvtx.empty)

_na = lambda v: not (isinstance(v, float) and v != v)  # not-nan check

opt_rows = [
    {
        "opportunity":        "Small GEMM / fragmented dense kernels",
        "evidence_found":     "yes",
        "supporting_metric":  (
            f"sgemm_32x32 ~{_dom_pct:.0f}% GPU time; avg {_sgemm_aus:.1f} us/call; "
            f"{_sm_all:,.0f} launches/slice"
        ),
        "affected_scenarios": "train, finetune, inference",
        "confidence":         "high",
        "next_experiment":    "NCU LaunchStats: Waves/SM vs 142-SM device; assess batching",
        "notes":              "TR-N1 NCU bundle captured; Waves/SM avg 0.617 observed",
    },
    {
        "opportunity":        "Flash attention hotspot (fmha_f32_64x64)",
        "evidence_found":     "yes" if _na(_fmha_pct) and _fmha_pct > 0 else "no",
        "supporting_metric":  (
            f"fmha_cutlassF ~{_fmha_pct:.1f}% GPU time (rank 2 all scenarios)"
            if _na(_fmha_pct) and _fmha_pct > 0 else "not detected"
        ),
        "affected_scenarios": "train, finetune, inference",
        "confidence":         "high",
        "next_experiment":    "NCU TR-N2: SpeedOfLight on fmha — confirm memory vs compute bound",
        "notes":              "Approved as NCU hotspot_2",
    },
    {
        "opportunity":        "High kernel launch count / overhead",
        "evidence_found":     "yes",
        "supporting_metric":  (
            f"cudaLaunchKernel ~71% API time; {_sm_all:,.0f} sgemm_32x32 launches/slice"
        ),
        "affected_scenarios": "train, finetune, inference",
        "confidence":         "medium",
        "next_experiment":    "Measure launch overhead as fraction of total wall time",
        "notes":              "Secondary to compute bottleneck; splitKreduce also high-count",
    },
    {
        "opportunity":        "cudaHostAlloc / optimizer-state allocation",
        "evidence_found":     "yes" if _h_elev else "partial",
        "supporting_metric":  (
            f"cudaHostAlloc avg {_ha_us:.0f} us/call in train/finetune"
            if _na(_ha_us) else "cudaHostAlloc below threshold or absent"
        ),
        "affected_scenarios": "train, finetune",
        "confidence":         "medium",
        "next_experiment":    "Profile with pinned-memory pool or pre-allocated optimizer buffers",
        "notes":              "Near-absent in inference (no optimizer); secondary priority",
    },
    {
        "opportunity":        "NVTX region imbalance",
        "evidence_found":     "partial" if _nvtx_ok else "unknown",
        "supporting_metric":  (
            "eval_task ~48%, train_epoch ~33%, final_test_pass ~17% in train"
            if _nvtx_ok else "NVTX data not parsed"
        ),
        "affected_scenarios": "train, finetune",
        "confidence":         "medium" if _nvtx_ok else "unknown",
        "next_experiment":    "Confirm eval_task dominance is expected at slice_size_tier=8/4",
        "notes":              "Slice size is minimal; production balance may differ",
    },
    {
        "opportunity":        "Sparse / graph aggregation bottleneck",
        "evidence_found":     "no",
        "supporting_metric":  "No SpMM / cusparse kernels in top-10 GPU kernels",
        "affected_scenarios": "none detected",
        "confidence":         "low",
        "next_experiment":    "Search full kernel list for cusparse / at::sparse kernels",
        "notes":              "Griffin is GNN-based but realistic slice uses dense GEMM path",
    },
    {
        "opportunity":        "DRAM bandwidth saturation",
        "evidence_found":     "partial",
        "supporting_metric":  "NCU TR-N1: DRAM throughput ~11.8% SoL; Memory Throughput ~32.9% SoL",
        "affected_scenarios": "train (NCU evidence); others inferred",
        "confidence":         "medium",
        "next_experiment":    "NCU MemoryWorkloadAnalysis across all 3 scenarios",
        "notes":              "Current evidence: compute-bound, not memory-bound, for sgemm_32x32",
    },
]

opt_matrix = pd.DataFrame(opt_rows)
display(opt_matrix)

## 8. Validation Against `RESULTS.md`

Comparing notebook-computed values against the claims in
`profiling/RESULTS.md → gfm-20260304-r02-realistic-cross-scenario-review-01`.

- **PASS** — computed value matches claim within tolerance
- **FAIL** — computed value contradicts claim
- **UNKNOWN** — raw artifact unavailable; claim cannot be verified

In [ ]:
val_rows = []

def _add(claim, computed, status, source, notes=""):
    val_rows.append({
        "claim":           claim,
        "computed_result": str(computed),
        "status":          status,
        "evidence_source": source,
        "notes":           notes,
    })

if kernel_df.empty:
    _add("All 9 runs loaded", f"0/9", "UNKNOWN", "artifact discovery", "No data loaded")
else:
    _add(
        "All 9 expected runs loaded",
        f"{len(loaded_runs)}/9",
        "PASS" if len(loaded_runs) == 9 else "FAIL",
        "artifact discovery",
    )

    # Top-3 kernel identities and rank ordering
    if not top3_table.empty:
        for rank_val, expected in enumerate(RESULTS_MD_TOP3, 1):
            sub_names = top3_table[top3_table["rank"] == rank_val]["kernel_base"].unique().tolist()
            computed  = sub_names[0] if len(sub_names) == 1 else f"MIXED({sub_names})"
            match     = (len(sub_names) == 1 and sub_names[0] == expected)
            note      = "stable across all 9 slices" if len(sub_names) == 1 else "not stable"
            _add(
                f"Rank {rank_val} kernel = {expected}",
                computed,
                "PASS" if match else ("FAIL" if sub_names else "UNKNOWN"),
                "metrics.json top_gpu_kernels",
                note,
            )

    # Time-share % per scenario
    for scenario in SCENARIO_ORDER:
        for rank_val, expected_pct in enumerate(RESULTS_MD_SCENARIO_PCT[scenario], 1):
            kernel_name = RESULTS_MD_TOP3[rank_val - 1]
            sub = kernel_df[
                (kernel_df["scenario"] == scenario) &
                (kernel_df["kernel_base"] == kernel_name)
            ]
            if sub.empty:
                _add(
                    f"{scenario} rank {rank_val} ({kernel_name[:28]}) ~= {expected_pct}%",
                    "no data",
                    "UNKNOWN",
                    "metrics.json",
                    "kernel not found in loaded runs",
                )
            else:
                comp_pct = sub["time_pct"].mean()
                status   = "PASS" if abs(comp_pct - expected_pct) <= 2.0 else "FAIL"
                _add(
                    f"{scenario} rank {rank_val} ({kernel_name[:28]}) ~= {expected_pct}%",
                    f"{comp_pct:.1f}%",
                    status,
                    "metrics.json (mean across 3 slices)",
                    "tolerance +/- 2%",
                )

    # sgemm_32x32 launch counts
    if not sgemm_launches.empty:
        for scenario, expected_launches in RESULTS_MD_SGEMM_LAUNCHES.items():
            sub = sgemm_launches[sgemm_launches["scenario"] == scenario]
            if sub.empty:
                _add(
                    f"{scenario} sgemm_32x32 launches/slice ~= {expected_launches}",
                    "no data", "UNKNOWN", "metrics.json"
                )
            else:
                comp = sub["launches"].mean()
                status = "PASS" if abs(comp - expected_launches) / max(expected_launches, 1) < 0.05 else "FAIL"
                _add(
                    f"{scenario} sgemm_32x32 launches/slice ~= {expected_launches}",
                    f"{comp:.0f}",
                    status,
                    "metrics.json launches",
                    "tolerance +/- 5%",
                )

validation_df = pd.DataFrame(val_rows)

def _vstyle(v):
    if v == "PASS":    return "background-color: #d4edda; color: #155724; font-weight: bold"
    if v == "FAIL":    return "background-color: #f8d7da; color: #721c24; font-weight: bold"
    if v == "UNKNOWN": return "background-color: #fff3cd; color: #856404"
    return ""

_applymap2 = getattr(validation_df.style, "map", None) or validation_df.style.applymap
display(_applymap2(_vstyle, subset=["status"]))

pass_n    = (validation_df["status"] == "PASS").sum()
fail_n    = (validation_df["status"] == "FAIL").sum()
unknown_n = (validation_df["status"] == "UNKNOWN").sum()
display(Markdown(
    f"**Validation summary**: {pass_n} PASS &nbsp; {fail_n} FAIL &nbsp; "
    f"{unknown_n} UNKNOWN &nbsp; / {len(validation_df)} claims checked"
))

## 9. Experiment Hook: Next Profiling Question

> **Does Griffin's cross-scenario hotspot pattern suggest underfilled small dense
> kernels that could benefit from schema-aware packing or batching?**

This section defines the hypothesis and next experiment.  No model, training
code, or Griffin source file is modified here.

In [ ]:
_hypothesis = (
    "The dominance of `ampere_sgemm_32x32_sliced1x4_tn` (~30% GPU time, ~22k launches/slice "
    "in train/finetune) combined with NCU TR-N1 evidence of low Waves Per SM (avg 0.617) "
    "and low achieved occupancy (~15%) suggests that Griffin launches many small-tile GEMM "
    "calls that fail to saturate the 142-SM device.  Schema-aware GEMM packing or batching "
    "could reduce launch count, improve SM occupancy, and reduce total GPU time."
)

_existing = [
    "`ampere_sgemm_32x32_sliced1x4_tn` is rank-1 across all 9 slices / all 3 scenarios",
    "NCU TR-N1: Waves Per SM avg 0.617; achieved occupancy avg 14.7%; issue slots busy ~30%",
    "NCU TR-N1 grid size averages 261 blocks for a 142-SM device -> sub-wave launches",
    "Scheduler No-Eligible 69% of cycles — significant idle time between warp issues",
    "Train/finetune have ~3x more sgemm_32x32 launches than inference (scales with eval graph evals)",
]

_needed = [
    "NCU TR-N2: SpeedOfLight on `fmha_cutlassF_f32_aligned_64x64_rf_sm80` (hotspot_2) — "
    "compare occupancy regime to sgemm_32x32",
    "GEMM shape analysis: extract M, N, K dims of sgemm_32x32 calls via nsys SQLite "
    "or torch.profiler",
    "Microbenchmark: batched GEMM vs individual calls at same total FLOPs on same device",
    "Code audit: identify Griffin GNN layers that emit many sgemm_32x32 calls per graph eval",
    "Check applicability of cuBLASLt grouped GEMM or torch._foreach_* APIs",
]

display(Markdown("### Hypothesis"))
display(Markdown(_hypothesis))
display(Markdown("### Existing Evidence"))
display(Markdown("\n".join(f"- {e}" for e in _existing)))
display(Markdown("### Additional Evidence Needed"))
display(Markdown("\n".join(f"- {e}" for e in _needed)))

checklist_df = pd.DataFrame([
    {"step": "1. Extract GEMM shapes from nsys SQLite",     "tool": "profiling/sql/manual_queries.sql",              "status": "TODO"},
    {"step": "2. NCU TR-N2 on fmha_f32_64x64",              "tool": "scripts/run_ncu_hotspot.sh train hotspot_2",    "status": "approved"},
    {"step": "3. NCU sgemm_32x32 in finetune + inference",   "tool": "scripts/run_ncu_hotspot.sh finetune hotspot_1", "status": "TODO"},
    {"step": "4. Batched GEMM microbenchmark",               "tool": "scripts/ new standalone script",                "status": "TODO"},
    {"step": "5. Griffin GNN layer audit for sgemm calls",   "tool": "grep + code review",                            "status": "TODO"},
])
display(Markdown("### Experiment Checklist"))
display(checklist_df)
display(Markdown("**Status**: `hypothesis_only` — no model or training code changes made."))

## 10. Optional Export

Set `EXPORT_ARTIFACTS = True` in the configuration cell and re-run this cell to
write key tables and a Markdown summary to
`artifacts/profiles/dashboard_exports/`.

In [ ]:
if EXPORT_ARTIFACTS:
    import os
    EXPORT_DIR.mkdir(parents=True, exist_ok=True)

    if not kernel_df.empty:
        kernel_df.to_csv(EXPORT_DIR / "kernel_data.csv", index=False)
    if "scenario_agg" in dir() and not scenario_agg.empty:
        scenario_agg.to_csv(EXPORT_DIR / "scenario_kernel_agg.csv", index=False)
    if "top3_table" in dir() and not top3_table.empty:
        top3_table.to_csv(EXPORT_DIR / "top3_cross_scenario.csv", index=False)
    if "opt_matrix" in dir() and not opt_matrix.empty:
        opt_matrix.to_csv(EXPORT_DIR / "optimization_matrix.csv", index=False)
    if "validation_df" in dir() and not validation_df.empty:
        validation_df.to_csv(EXPORT_DIR / "validation_results.csv", index=False)
    if "avail_df" in dir():
        avail_df.to_csv(EXPORT_DIR / "artifact_availability.csv", index=False)

    _sum = summary_lines if "summary_lines" in dir() else ["Summary unavailable."]
    _val = (validation_df.to_markdown(index=False)
            if "validation_df" in dir() and not validation_df.empty else "Not available.")
    _opt = (opt_matrix.to_markdown(index=False)
            if "opt_matrix" in dir() and not opt_matrix.empty else "Not available.")

    md_text = (
        "# Griffin Cross-Scenario Dashboard Summary\n\n"
        "Generated from: profiling/notebooks/nsys_cross_scenario_dashboard.ipynb\n\n"
        "## Advisor Summary\n\n"
        + "\n".join(_sum)
        + "\n\n## Validation Results\n\n"
        + _val
        + "\n\n## Optimization Opportunity Matrix\n\n"
        + _opt
    )
    (EXPORT_DIR / "dashboard_summary.md").write_text(md_text)
    print(f"Exported to: {EXPORT_DIR.relative_to(REPO_ROOT)}")
    for f in sorted(EXPORT_DIR.iterdir()):
        print(f"  {f.name}")
else:
    display(Markdown(
        "Export disabled.  Set `EXPORT_ARTIFACTS = True` in the config cell "
        "and re-run this cell to write tables and summary to "
        "`artifacts/profiles/dashboard_exports/`."
    ))